In [5]:
import ultralytics
ultralytics.checks()

Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24135MiB)
Setup complete ✅ (64 CPUs, 125.6 GB RAM, 1644.5/1831.2 GB disk)


In [14]:
from roboflow import Roboflow
import os

rf = Roboflow(api_key="hsQkPkeB5bwobCxIg7Ew")

project_det = rf.workspace("detection-fe3wj").project("mil_vechicle_segmentation-69ull")
dataset_det = project_det.version(2).download("yolov8")

yaml_path = os.path.join(dataset_det.location, 'data.yaml')
print(f"✅ Датасет завантажено. Шлях до конфігу: {yaml_path}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Mil_Vechicle_Segmentation-2 in yolov8:: 100%|██████████| 4730/4730 [00:00<00:00, 11924.72it/s]


✅ Датасет завантажено. Шлях до конфігу: /home/obahlai/Documents/detection_segmentation/Mil_Vechicle_Segmentation-2/data.yaml


In [16]:
from ultralytics import YOLO

print(f"\n{'='*20} ЕТАП 1: ОСНОВНЕ ТРЕНУВАННЯ {'='*20}\n")

model = YOLO('yolov8n-seg.pt')

results = model.train(
    data=yaml_path,       
    epochs=300,          
    patience=50,         
    imgsz=640,           
    
    device=[0, 1, 2],    
    batch=48,          
    workers=8,           
    name="yolov8n_seg_phase1_multi", 
    
    mosaic=1.0,          
    
    mixup=0.0, degrees=0.0, translate=0.0, 
    scale=0.0, fliplr=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, copy_paste=0.0,
    
    optimizer='AdamW',   
    lr0=0.001,           
    cos_lr=True,         
    weight_decay=0.0005, 
    
    box=7.5, dfl=1.5, 
    exist_ok=True, save=True      
)


==================== ЕТАП 1: ОСНОВНЕ ТРЕНУВАННЯ ====================

New https://pypi.org/project/ultralytics/8.4.26 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24135MiB)
                                                      CUDA:1 (NVIDIA GeForce RTX 3090, 24132MiB)
                                                      CUDA:2 (NVIDIA GeForce RTX 3090, 24135MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=48, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/home/obahlai/Documents/detection_segmentation/Mil_Vechicle_Segmentation-2/data.yaml, degrees=0.0, deterministic=True, device=0,1,2, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=t

In [17]:
from ultralytics import YOLO

print(f"\n{'='*20} ЕТАП 2: FINE-TUNING {'='*20}\n")

best_model_path = '/home/obahlai/Documents/detection_segmentation/runs/segment/yolov8n_seg_phase1_multi/weights/best.pt'
model = YOLO(best_model_path)

results = model.train(
    data='/home/obahlai/Documents/detection_segmentation/Mil_Vechicle_Segmentation-2/data.yaml',       
    epochs=30,           
    patience=30,         
    imgsz=640,           
    
    device=0,    
    batch=32,            
    workers=8,           
    name="yolov8n_seg_phase2_tuned", 
    
    optimizer='AdamW',   
    lr0=0.0001,          
    cos_lr=False,        
    
    mosaic=0.0, mixup=0.0, degrees=0.0, translate=0.0, 
    scale=0.0, fliplr=0.0, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, copy_paste=0.0,
    
    box=7.5, dfl=1.5, 
    exist_ok=True, save=True      
)

print(f"\n{'='*20} ТЮНІНГ ЗАВЕРШЕНО {'='*20}\n")


==================== ЕТАП 2: FINE-TUNING ====================

New https://pypi.org/project/ultralytics/8.4.26 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24135MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/obahlai/Documents/detection_segmentation/Mil_Vechicle_Segmentation-2/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, 

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import os

print(f"\n{'='*20} ГЕНЕРАЦІЯ АНАЛІТИКИ ДЛЯ ДИПЛОМУ {'='*20}\n")

# 1. Вказуємо шляхи до файлів results.csv обох етапів
path_phase1 = '/home/obahlai/Documents/detection_segmentation/runs/segment/yolov8n_seg_phase1_multi/results.csv'
path_phase2 = '/home/obahlai/Documents/detection_segmentation/runs/segment/yolov8n_seg_phase2_tuned/results.csv'

# 2. Зчитуємо дані (YOLO часто додає пробіли в назви колонок, тому очищаємо їх)
try:
    df1 = pd.read_csv(path_phase1)
    df2 = pd.read_csv(path_phase2)
    
    df1.columns = df1.columns.str.strip()
    df2.columns = df2.columns.str.strip()
except FileNotFoundError as e:
    print(f"❌ Помилка: Не знайдено файл результатів. {e}")
    print("Переконайтеся, що обидва етапи тренування завершені.")
    raise

# 3. Налаштовуємо нумерацію епох, щоб графік Етапу 2 продовжував графік Етапу 1
epochs_phase1 = len(df1)
df1['epoch'] = range(1, epochs_phase1 + 1)
df2['epoch'] = range(epochs_phase1 + 1, epochs_phase1 + len(df2) + 1)

# 4. Створюємо полотно для графіка
plt.figure(figsize=(12, 6), dpi=300)

# Будуємо лінію для Етапу 1 (синя)
plt.plot(df1['epoch'], df1['metrics/mAP50(M)'], 
         label='Етап 1: Основне навчання (AdamW, LR=0.001)', 
         color='#1f77b4', linewidth=2.5)

# Будуємо лінію для Етапу 2 (зелена)
plt.plot(df2['epoch'], df2['metrics/mAP50(M)'], 
         label='Етап 2: Fine-Tuning (LR=0.0001)', 
         color='#2ca02c', linewidth=2.5)

# Вертикальна червона лінія, що позначає момент зміни стратегії
plt.axvline(x=epochs_phase1, color='#d62728', linestyle='--', linewidth=2, 
            label='Момент зміни Learning Rate')

# 5. Оформлення (підписи, сітка, легенда)
plt.title('Динаміка точності сегментації масок (mAP@50) під час двоетапного навчання', 
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Епохи навчання', fontsize=12)
plt.ylabel('mAP@50 (Mask)', fontsize=12)

# Додаємо сітку для кращої читабельності
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend(loc='lower right', fontsize=10, shadow=True)

# 6. Зберігаємо результат у високій якості для вставки в Word/PowerPoint
output_image = '/home/obahlai/Documents/detection_segmentation/diploma_mAP_comparison.png'
plt.savefig(output_image, bbox_inches='tight')

# Показуємо графік у Jupyter
plt.show()

print(f"✅ Графік успішно згенеровано та збережено за шляхом:\n{output_image}")


==================== ГЕНЕРАЦІЯ АНАЛІТИКИ ДЛЯ ДИПЛОМУ ====================



<Figure size 3600x1800 with 1 Axes>

✅ Графік успішно згенеровано та збережено за шляхом:
/home/obahlai/Documents/detection_segmentation/diploma_mAP_comparison.png


In [19]:
from ultralytics import YOLO
import pandas as pd

print(f"\n{'='*20} ВАЛІДАЦІЯ ТА ПОРІВНЯННЯ МОДЕЛЕЙ {'='*20}\n")

yaml_path = '/home/obahlai/Documents/detection_segmentation/Mil_Vechicle_Segmentation-2/data.yaml'
model1_path = '/home/obahlai/Documents/detection_segmentation/runs/segment/yolov8n_seg_phase1_multi/weights/best.pt'
model2_path = '/home/obahlai/Documents/detection_segmentation/runs/segment/yolov8n_seg_phase2_tuned/weights/best.pt'

# 1. Валідація моделі Етапу 1
print("⏳ Тестування моделі після Етапу 1 (Основне навчання)...")
model1 = YOLO(model1_path)
val1 = model1.val(data=yaml_path, imgsz=640, device=0, split='val', verbose=False)

# 2. Валідація моделі Етапу 2
print("\n⏳ Тестування моделі після Етапу 2 (Fine-Tuning)...")
model2 = YOLO(model2_path)
val2 = model2.val(data=yaml_path, imgsz=640, device=0, split='val', verbose=False)

# 3. Витягуємо метрики у зручний формат
def extract_metrics(val_results, model_name):
    # Беремо метрики для масок (M) та рамок (B)
    return {
        "Етап": model_name,
        "Precision (M)": round(val_results.seg.mp, 3),
        "Recall (M)": round(val_results.seg.mr, 3),
        "mAP@50 (Box)": round(val_results.box.map50, 3),
        "mAP@50-95 (Box)": round(val_results.box.map, 3),
        "mAP@50 (Mask)": round(val_results.seg.map50, 3),
        "mAP@50-95 (Mask)": round(val_results.seg.map, 3)
    }

data = [
    extract_metrics(val1, "Етап 1 (Базова)"),
    extract_metrics(val2, "Етап 2 (Tuned)")
]

# 4. Створюємо таблицю (DataFrame)
df = pd.DataFrame(data)

# Рахуємо різницю (буст) для масок
boost = round((df.loc[1, 'mAP@50 (Mask)'] - df.loc[0, 'mAP@50 (Mask)']) * 100, 2)

print(f"\n{'='*20} ФІНАЛЬНА ТАБЛИЦЯ ДЛЯ ДИПЛОМУ {'='*20}\n")
display(df) # В Jupyter це намалює гарну табличку

print(f"\n📈 Абсолютний приріст mAP@50 (Mask) після тюнінгу: +{boost}%")
print("Збережіть ці дані для пояснювальної записки!")


==================== ВАЛІДАЦІЯ ТА ПОРІВНЯННЯ МОДЕЛЕЙ ====================

⏳ Тестування моделі після Етапу 1 (Основне навчання)...
Ultralytics 8.3.241 🚀 Python-3.12.3 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24135MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1145.0±332.8 MB/s, size: 17.0 KB)
val: Scanning /home/obahlai/Documents/detection_segmentation/Mil_Vechicle_Segmentation-2/valid/labels.cache... 197 images, 36 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 197/197 334.4Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 6.5it/s 2.0s<0.2s
                   all        197        573       0.83      0.637       0.75      0.503      0.823      0.635      0.746      0.512
Speed: 1.7ms preprocess, 2.0ms inference, 0.0ms loss, 1.5ms postprocess per image
Results saved to 

,Етап,Precision (M),Recall (M),mAP@50 (Box),mAP@50-95 (Box),mAP@50 (Mask),mAP@50-95 (Mask)
0,Етап 1 (Базова),0.823,0.635,0.750,0.503,0.746,0.512
1,Етап 2 (Tuned),0.819,0.631,0.737,0.505,0.748,0.515



📈 Абсолютний приріст mAP@50 (Mask) після тюнінгу: +0.2%
Збережіть ці дані для пояснювальної записки!
